In [1]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import f1_score, precision_recall_fscore_support, confusion_matrix, roc_auc_score


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 5 (Legal Statute Identification, MULTI-LABEL): KNOWLEDGE DISTILLATION
#
#   TEACHER = Phase-1 Architecture-4 hybrid prefix-LM (205.5M weights, the LARGEST Phase-1 model)
#             + classifier, trained on LSI in this notebook (STEP 1, CELL 13)
#   STUDENT = Phase-1 Architecture-2 encoder (~68M weights used, the smallest and fastest)
#             + the same classifier, trained with the teacher's probabilities (STEP 3, CELL 15)
#             -> submitted
#
#   Manual: "use a larger or stronger teacher model and transfer its prediction knowledge
#   to a smaller student model". Both start from Phase-1 models; LSI data only.
#
#   both read the same input: up to 8 chunks of 512 tokens per document
#   both use the same classifier (as the other Phase-5 notebooks): mix of ALL layers ->
#        LINEAR path (starts as a logistic regression on the frozen features, CELL 11)
#        + LABEL ATTENTION (every statute attends to all tokens of the document, starts at 0)
#   teacher: Phase-1 model frozen, only the classifier trains (the 205M model runs without gradients)
#   student: classifier + all Architecture-2 encoder layers (layer-wise learning rates)
#   student loss = (1 - KD_ALPHA) * BCE(gold statutes)
#                  + KD_ALPHA * T^2 * BCE(student, teacher probabilities of all 100 statutes)
#   the teacher scores train documents it did NOT learn from (honest soft labels)
#
# Trained on TRAIN only; dev picks the best epochs and the thresholds.
# ============================================================

RUN_NAME = "p5_kd_arch4_to_arch2"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 checkpoints. If a path is wrong, CELL 9 searches /kaggle/input for it.
TEACHER_CHECKPOINT = Path(       # Architecture-4 (HybridLM) best.pt
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-4/model4_checkpoints/best.pt"
)
STUDENT_CHECKPOINT = Path(       # Architecture-2 (encoder-decoder) best_encoder_decoder.pt
    "/kaggle/input/notebooks/ritikguptatab/proj-cap-p-1-archi-2/architecture_2_encoder_decoder/best_encoder_decoder.pt"
)

# Phase-5 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-5-lsi/Phase_5_LSI")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Input (same for teacher and student)
# ------------------------------------------------------------

CHUNK_LEN = 512
MAX_CHUNKS = 8
PLACEHOLDERS = {"<ENTITY>": "X", "<SECTION>": "section", "<ACT>": "Act"}
POOL_SKIP_SPACES = True      # whitespace tokens are not pooled / attended (they stay in the input)


# ------------------------------------------------------------
# Knowledge distillation (Hinton, Vinyals & Dean, 2015)
# ------------------------------------------------------------

KD_ALPHA = 0.5               # weight of the teacher term (0 = no distillation)
KD_TEMPERATURE = 1.0         # the teacher's probabilities are already soft

# the train set is split once: the teacher learns from TEACHER_TRAIN_DOCS documents, then scores
# STUDENT_TRAIN_DOCS OTHER train documents; the student learns from those (gold + teacher)
TEACHER_TRAIN_DOCS = 16000
STUDENT_TRAIN_DOCS = 16000   # ~40 min for the teacher to score them


# ------------------------------------------------------------
# Head (same for teacher and student)
# ------------------------------------------------------------

POOL_SKIP_TOKENS = 4
LABEL_ATTENTION_DIM = 256
HEAD_DROPOUT = 0.1
LR_HEAD = 1e-3

POS_WEIGHT_POWER = 0.5
POS_WEIGHT_MAX = 10.0


# ------------------------------------------------------------
# What trains in the Phase-1 models
# ------------------------------------------------------------

STUDENT_TOP_BLOCKS = 6       # Architecture-2 encoder layers that train (6 = all; the embeddings stay frozen)
LR_TOP_BLOCK = 3e-5          # top encoder layer; every layer below gets LAYER_LR_DECAY x the rate above it
LAYER_LR_DECAY = 0.8

BACKBONE_DROPOUT = 0.0       # Phase-1 dropout inside both models is switched off
PREFIX_LENGTH = None         # Architecture-4 bidirectional prefix: None = the Phase-1 prefix_length


# ------------------------------------------------------------
# Training (TRAIN data only)
#   Architecture-4 needs ~3x the compute of the Architecture-2 encoder per document
#   epoch 0 = the start (logistic regression), kept if no epoch beats it
# ------------------------------------------------------------

TEACHER_EPOCHS = 2           # classifier only (the teacher's Phase-1 model stays frozen)
TEACHER_EPOCH_DOCS = 8000
TEACHER_BATCH_DOCS = 2
TEACHER_GRAD_ACCUM = 8       # -> 16 documents per weight update

STUDENT_EPOCHS = 4           # epoch 1 = classifier only, then classifier + encoder layers
STUDENT_EPOCH_DOCS = 8000
STUDENT_BATCH_DOCS = 2       # If CUDA out-of-memory: 1
STUDENT_GRAD_ACCUM = 8
STUDENT_HEAD_ONLY_EPOCHS = 1

PATIENCE = 2
WARMUP_RATIO = 0.05
GRAD_CLIP = 1.0

EVAL_CHUNKS = 24             # chunks per forward pass when predicting
DEV_EVAL_DOCS = 2000         # dev documents scored after every epoch (the full dev set at the end, CELL 16)

# Before each training (CELL 11): probe of every layer + start of the linear path on
# PROBE_DOCS train documents, token statistics from STATS_DOCS train documents.
STATS_DOCS = 500
PROBE_DOCS = 3000


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 200             # per split

if DEBUG_MODE:
    TEACHER_EPOCHS = STUDENT_EPOCHS = 2
    TEACHER_EPOCH_DOCS = STUDENT_EPOCH_DOCS = 64
    TEACHER_TRAIN_DOCS = STUDENT_TRAIN_DOCS = 100
    PROBE_DOCS = 100
    STATS_DOCS = 50
    DEV_EVAL_DOCS = 100


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


print("Run       :", RUN_NAME)
print("Reads     :", f"up to {MAX_CHUNKS} chunks x {CHUNK_LEN} tokens per document")
print("KD        :", f"teacher Architecture-4 -> student Architecture-2 encoder | alpha {KD_ALPHA}, T {KD_TEMPERATURE}")
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")


Run       : p5_kd_arch4_to_arch2
Reads     : up to 8 chunks x 512 tokens per document
KD        : teacher Architecture-4 -> student Architecture-2 encoder | alpha 0.5, T 1.0
Output    : /kaggle/working/p5_kd_arch4_to_arch2
AMP dtype : torch.float16


In [3]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER + FAST ENCODING
#
# Documents are long, so only the parts we need are tokenized,
# and every pre-token piece is BPE-encoded only once (cache).
# Placeholders (<ENTITY>, <SECTION>, <ACT>) are replaced first.
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


piece_cache = {}      # pre-token piece -> token ids


def encode_fast(text):
    """Same ids as tokenizer.encode(text), but every piece is encoded only once."""

    ids = []

    for piece in tok_module.PRETOKEN_PATTERN.findall(tok_module.normalize_text(text)):

        if piece not in piece_cache:
            piece_cache[piece] = tokenizer.encode(piece)

        ids.extend(piece_cache[piece])

    return ids


def head_tokens(text, n):
    """First n tokens of text (tokenizes only the beginning, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[:n]

        part = text[:chars]
        part = part[:max(part.rfind(" "), 1)]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[:n]

        chars *= 2


def tail_tokens(text, n):
    """Last n tokens of text (tokenizes only the end, cut at a space so no word is split)."""

    chars = n * 8

    while True:

        if chars >= len(text):
            return encode_fast(text)[-n:]

        part = text[-chars:]
        part = part[max(part.find(" "), 0):]

        ids = encode_fast(part)
        if len(ids) >= n:
            return ids[-n:]

        chars *= 2


demo = "The appeal is allowed. The order of the High Court dated 12.03.2010 is set aside."
assert encode_fast(demo) == tokenizer.encode(demo)
print("Demo:", [tokenizer.id_to_token[i] for i in encode_fast(demo)][:12])

def clean(text):
    """Placeholders -> short ordinary words (PLACEHOLDERS, CELL 2)."""
    for old, new in PLACEHOLDERS.items():
        text = text.replace(old, new)
    return text


print("\nHow the placeholders are tokenized:")
for old, new in PLACEHOLDERS.items():
    print(f"  {old:10s} -> {len(encode_fast(old))} tokens {[tokenizer.id_to_token[i] for i in encode_fast(old)]} | "
          f"replaced by {new!r} -> {len(encode_fast(' ' + new))} token(s)")


# whitespace-only tokens (" ", "\n", ...): kept in the input, not pooled (POOL_SKIP_SPACES)
def token_text(i):
    try:
        return tokenizer.id_to_token[i]
    except (KeyError, IndexError):
        return None


SPACE_TOKEN_IDS = [i for i in range(VOCAB_SIZE)
                   if i != PAD_ID and isinstance(token_text(i), str) and token_text(i).strip() == ""]

print(f"\nWhitespace-only tokens in the vocabulary: {len(SPACE_TOKEN_IDS)}, e.g. "
      f"{[token_text(i) for i in SPACE_TOKEN_IDS[:6]]}")


Vocabulary size: 32004 | PAD / UNK: 0 1
Demo: ['The', ' ', 'appeal', ' ', 'is', ' ', 'allow', 'ed', '.', ' ', 'The', ' ']

How the placeholders are tokenized:
  <ENTITY>   -> 5 tokens ['<', 'EN', 'TI', 'TY', '>'] | replaced by 'X' -> 2 token(s)
  <SECTION>  -> 3 tokens ['<', 'SECTION', '>'] | replaced by 'section' -> 2 token(s)
  <ACT>      -> 3 tokens ['<', 'ACT', '>'] | replaced by 'Act' -> 2 token(s)

Whitespace-only tokens in the vocabulary: 3, e.g. [' ', '\n', '\n\n']


In [4]:
# ============================================================
# CELL 4 — PHASE-5 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE5_FILES = {
    "train": "Phase_5_LSI/Train/train.json",
    "dev":   "Phase_5_LSI/Train/dev.json",
    "test":  "Phase_5_LSI/Test/test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE5_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_5_LSI/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE5_FILES.items():
        DATA_FILES[name] = hf_hub_download(repo_id=HF_DATASET_REPO, filename=filename,
                                           repo_type="dataset", token=HF_TOKEN)

for name, path in DATA_FILES.items():
    print(f"{name:6s} -> {path}  ({os.path.getsize(path) / 1e6:.0f} MB)")


train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/train.json  (299 MB)
dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Train/dev.json  (79 MB)
test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/93b0059be756c6c461f3a225de25aabda7d5f84a/Phase_5_LSI/Test/test.json  (98 MB)


In [5]:
# ============================================================
# CELL 5 — LOAD DOCUMENTS + STATUTES
#
# Every document: {"id", "text", "labels" (sorted, unique statute ids; None for test),
#                  "y" (0/1 vector over the statutes)}
# ============================================================

def read_json(path):
    """A JSON file, or a JSON-lines file (one document per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_docs(path, with_labels=True):

    docs = []

    for i, item in enumerate(read_json(path)):
        docs.append({
            "id": str(item.get("id", i)),
            "text": str(item.get("text", "") or ""),
            "labels": sorted(set(int(l) for l in item.get("labels", []))) if with_labels else None,
        })

    if DEBUG_MODE:
        docs = docs[:DEBUG_DOCS]

    return docs


train_docs = load_docs(DATA_FILES["train"])
dev_docs = load_docs(DATA_FILES["dev"])
test_docs = load_docs(DATA_FILES["test"], with_labels=False)     # test labels are private: never read

ids = [d["id"] for d in test_docs]
assert len(ids) == len(set(ids)), "test ids are not unique"


# ------------------------------------------------------------
# Statutes: the label columns
# ------------------------------------------------------------

LABELS = sorted({l for d in train_docs for l in d["labels"]})        # statute ids, e.g. 0 .. 99
NUM_LABELS = len(LABELS)
LABEL_INDEX = {l: j for j, l in enumerate(LABELS)}

missing = sorted({l for d in dev_docs for l in d["labels"]} - set(LABELS))
assert not missing, f"dev statutes never seen in train: {missing}"

for d in train_docs + dev_docs:
    d["y"] = np.zeros(NUM_LABELS, dtype=np.float32)
    d["y"][[LABEL_INDEX[l] for l in d["labels"]]] = 1.0

Y_TRAIN = np.stack([d["y"] for d in train_docs])
POSITIVES = Y_TRAIN.sum(axis=0)

# weight on the positives of every statute (BCE): (negatives / positives) ** POWER
POS_WEIGHT = torch.tensor(np.clip(((len(train_docs) - POSITIVES) / np.maximum(POSITIVES, 1)) ** POS_WEIGHT_POWER,
                                  1.0, POS_WEIGHT_MAX), dtype=torch.float32)

print(f"{'split':6s} {'docs':>7s} {'statutes per doc':>17s} {'chars (median / 90%)':>22s}")
for name, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    chars = np.array([len(d["text"]) for d in docs])
    per_doc = np.mean([len(d["labels"]) for d in docs]) if docs[0]["labels"] is not None else float("nan")
    print(f"{name:6s} {len(docs):7,d} {per_doc:17.2f} {np.median(chars):11,.0f} / {np.percentile(chars, 90):8,.0f}")

print(f"\n{NUM_LABELS} statutes (ids {LABELS[0]} .. {LABELS[-1]}) | train documents per statute: "
      f"min {int(POSITIVES.min())}, median {int(np.median(POSITIVES))}, max {int(POSITIVES.max())}")
print(f"positive weights: min {POS_WEIGHT.min():.2f}, median {POS_WEIGHT.median():.2f}, max {POS_WEIGHT.max():.2f}")


split     docs  statutes per doc   chars (median / 90%)
train   42,750              3.70       2,717 /   16,789
dev     10,181              3.92       3,418 /   17,900
test    13,019               nan       3,368 /   17,277

100 statutes (ids 0 .. 99) | train documents per statute: min 272, median 679, max 11730
positive weights: min 1.63, median 7.84, max 10.00


In [6]:
# ============================================================
# CELL 6 — TOKENIZE + CHUNKS
#
#   whole document fits in MAX_CHUNKS x CHUNK_LEN tokens -> its consecutive chunks
#   longer document -> MAX_CHUNKS chunks spread over it: the text is cut into
#   MAX_CHUNKS equal parts and the first CHUNK_LEN tokens of every part are kept
# ============================================================

MAX_TOKENS = MAX_CHUNKS * CHUNK_LEN


def make_chunks(text):
    """-> (list of token-id arrays, True if the document was too long to read completely)"""

    text = clean(text)

    if len(text) <= MAX_TOKENS * 4:                      # short enough to tokenize completely (~2.3 chars per token)

        ids = encode_fast(text) or encode_fast(".")      # (an empty document becomes ".")

        if len(ids) <= MAX_TOKENS:
            chunks = [ids[k:k + CHUNK_LEN] for k in range(0, len(ids), CHUNK_LEN)]
            return [np.array(c, dtype=np.int32) for c in chunks], False

    # long document: MAX_CHUNKS chunks spread over its start, middle and end
    step = len(text) / MAX_CHUNKS
    chunks = [head_tokens(text[int(k * step):int((k + 1) * step)], CHUNK_LEN) for k in range(MAX_CHUNKS)]

    return [np.array(c, dtype=np.int32) for c in chunks if len(c)], True


t0 = time.time()

for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:

    for k, d in enumerate(docs):
        d["chunks"], d["cut"] = make_chunks(d["text"])
        del d["text"]                                    # free memory (the text is not needed any more)

        if (k + 1) % 10000 == 0:
            print(f"  {split}: {k + 1}/{len(docs)} documents | {time.time() - t0:.0f}s", flush=True)

print(f"Tokenized in {time.time() - t0:.0f}s")

n_chunks = Counter(len(d["chunks"]) for d in train_docs)
print(f"\nChunks per train document: {dict(sorted(n_chunks.items()))}")
for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    print(f"{split:6s}: mean {np.mean([len(d['chunks']) for d in docs]):.2f} chunks per document | "
          f"read completely {1 - np.mean([d['cut'] for d in docs]):.1%} | "
          f"longer than {MAX_TOKENS} tokens ({MAX_CHUNKS} chunks spread over the document) {np.mean([d['cut'] for d in docs]):.1%}")

sample_ids = np.concatenate([c for d in train_docs[:2000] for c in d["chunks"]])
print(f"\nwhitespace tokens: {np.isin(sample_ids, SPACE_TOKEN_IDS).mean():.1%} of all tokens "
      f"({'not pooled' if POOL_SKIP_SPACES else 'pooled'})")


  train: 10000/42750 documents | 15s
  train: 20000/42750 documents | 28s
  train: 30000/42750 documents | 41s
  train: 40000/42750 documents | 54s
  dev: 10000/10181 documents | 73s
  test: 10000/13019 documents | 88s
Tokenized in 92s

Chunks per train document: {1: 11500, 2: 9303, 3: 4129, 4: 2541, 5: 1891, 6: 1558, 7: 1397, 8: 10431}
train : mean 3.85 chunks per document | read completely 78.5% | longer than 4096 tokens (8 chunks spread over the document) 21.5%
dev   : mean 4.16 chunks per document | read completely 75.3% | longer than 4096 tokens (8 chunks spread over the document) 24.7%
test  : mean 4.11 chunks per document | read completely 76.9% | longer than 4096 tokens (8 chunks spread over the document) 23.1%

whitespace tokens: 39.6% of all tokens (not pooled)


In [7]:
# ============================================================
# CELL 7 — METRICS + THRESHOLDS (multi-label)
#   macro-F1 (the competition metric: mean F1 over the statutes), micro-F1, subset accuracy
#   one threshold per statute = the one with the best F1 for that statute on dev
#   "at least one": a document with no statute above its threshold gets its most probable one
# ============================================================

def multilabel_scores(Y, P):
    """Y, P: [docs, statutes] 0/1 arrays."""

    Y, P = Y.astype(bool), P.astype(bool)
    tp = (Y & P).sum(axis=0)
    fp = (~Y & P).sum(axis=0)
    fn = (Y & ~P).sum(axis=0)

    f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)

    return {
        "macro_f1": float(f1.mean()),
        "micro_f1": float(2 * tp.sum() / max(2 * tp.sum() + fp.sum() + fn.sum(), 1)),
        "subset_acc": float((Y == P).all(axis=1).mean()),
        "macro_precision": float(np.mean(tp / np.maximum(tp + fp, 1))),
        "macro_recall": float(np.mean(tp / np.maximum(tp + fn, 1))),
        "per_statute_f1": f1,
    }


THRESHOLD_GRID = np.linspace(0.02, 0.98, 49)


def tune_thresholds(Y, probs):
    """One threshold per statute: the one with the best F1 on these documents."""

    Y = Y.astype(bool)
    best_f1 = np.full(Y.shape[1], -1.0)
    best_t = np.full(Y.shape[1], 0.5)

    for t in THRESHOLD_GRID:
        P = probs >= t
        tp = (Y & P).sum(axis=0)
        f1 = 2 * tp / np.maximum(P.sum(axis=0) + Y.sum(axis=0), 1)
        better = f1 > best_f1
        best_f1[better], best_t[better] = f1[better], t

    return best_t


def decide(probs, thresholds, at_least_one):
    P = probs >= thresholds
    if at_least_one:
        empty = ~P.any(axis=1)
        P[np.where(empty)[0], probs[empty].argmax(axis=1)] = True
    return P


def dev_scores(Y, probs):
    """Thresholds tuned on these documents; returns scores, thresholds, at_least_one."""

    thresholds = tune_thresholds(Y, probs)
    best = None

    for at_least_one in (False, True):
        s = multilabel_scores(Y, decide(probs, thresholds, at_least_one))
        if best is None or s["macro_f1"] > best[0]["macro_f1"]:
            best = (s, thresholds, at_least_one)

    return best


# check on a tiny example
_Y = np.array([[1, 0], [0, 1], [1, 1]])
_s = multilabel_scores(_Y, np.array([[1, 0], [0, 0], [1, 1]]))
assert abs(_s["macro_f1"] - (1.0 + 2 / 3) / 2) < 1e-9 and abs(_s["subset_acc"] - 2 / 3) < 1e-9
print("✅ metrics check passed")


✅ metrics check passed


In [8]:
# ============================================================
# CELL 8 — MODELS
#
#   TEACHER backbone: Architecture-4 hybrid prefix-LM     (Phase-1 weights, frozen)
#   STUDENT backbone: Architecture-2 ENCODER               (Phase-1 weights, encoder layers train)
#   same classifier on both:
#     every chunk -> backbone -> token vector = learned mix of ALL layers -> standardized
#     LINEAR path    : mean of the document's token vectors -> 100 statute scores
#                      (starts as the logistic regression of CELL 11)
#     LABEL ATTENTION: every statute attends to all tokens of the document -> 100 scores (starts at 0)
#
# Layer names match the Phase-1 notebooks, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-4 HYBRID PREFIX-LM (bidirectional prefix branch + causal branch, gated fusion)
# ============================================================

class HybridRMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        xf = x.float()
        return (self.weight * (xf * torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps))).to(x.dtype)


class HybridAttention(nn.Module):
    """Used twice per block: bidirectional (prefix / encoder branch) and causal (causal branch)."""

    def __init__(self, d_model, n_heads, dropout, causal):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.causal = causal

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, attn_mask):

        B, T, D = x.shape
        q, k, v = (p(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
                   for p in (self.q_proj, self.k_proj, self.v_proj))

        p = self.dropout.p if self.training else 0.0

        if self.causal:      # padding is at the end, so real tokens never see it
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=p)
        else:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=p)

        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, D))


class GatedHybridFusion(nn.Module):

    def __init__(self, d_model):
        super().__init__()
        self.gate_proj = nn.Linear(2 * d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, encoder_output, causal_output):
        gate = torch.sigmoid(self.gate_proj(torch.cat([encoder_output, causal_output], dim=-1)))
        return self.out_proj(gate * encoder_output + (1.0 - gate) * causal_output)


class HybridSwiGLU(nn.Module):

    def __init__(self, d_model, ffn_dim, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.up_proj = nn.Linear(d_model, ffn_dim, bias=False)
        self.down_proj = nn.Linear(ffn_dim, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class HybridTransformerBlock(nn.Module):

    def __init__(self, cfg):
        super().__init__()

        d, h, p, eps = cfg["d_model"], cfg["n_heads"], cfg["dropout"], cfg["norm_eps"]

        self.encoder_norm = HybridRMSNorm(d, eps)
        self.causal_norm = HybridRMSNorm(d, eps)
        self.encoder_attention = HybridAttention(d, h, p, causal=False)
        self.causal_attention = HybridAttention(d, h, p, causal=True)
        self.hybrid_fusion = GatedHybridFusion(d)
        self.ffn_norm = HybridRMSNorm(d, eps)
        self.ffn = HybridSwiGLU(d, cfg["ffn_dim"], p)

    def forward(self, x, mask, prefix):

        T = x.size(1)

        # encoder branch: the first `prefix` tokens attend to each other (both directions);
        # tokens after the prefix get 0 from this branch (as in pretraining)
        encoder_output = self.encoder_attention(self.encoder_norm(x[:, :prefix]), mask[:, None, None, :prefix])
        encoder_output = F.pad(encoder_output, (0, 0, 0, T - prefix))

        causal_output = self.causal_attention(self.causal_norm(x), None)

        x = x + self.hybrid_fusion(encoder_output, causal_output)

        return x + self.ffn(self.ffn_norm(x))


class Arch4Hybrid(nn.Module):
    """Architecture-4 HybridLM without the LM head: tokens -> hidden vectors."""

    def __init__(self, cfg):
        super().__init__()

        self.prefix_length = cfg["prefix_length"]

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = nn.Embedding(cfg["block_size"], cfg["d_model"])
        self.embedding_dropout = nn.Dropout(cfg["dropout"])
        self.blocks = nn.ModuleList([HybridTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        self.final_norm = HybridRMSNorm(cfg["d_model"], cfg["norm_eps"])

    def forward(self, input_ids, mask, all_layers=False):
        """all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every block (before any norm)."""

        T = input_ids.size(1)
        prefix = min(self.prefix_length, T)

        positions = torch.arange(T, device=input_ids.device)
        x = self.embedding_dropout(self.token_embedding(input_ids) + self.position_embedding(positions))
        states = [x]

        for block in self.blocks:
            x = block(x, mask, prefix)
            states.append(x)

        if all_layers:
            return states

        return self.final_norm(x)


# ============================================================
# Architecture-2 ENCODER (STUDENT backbone)
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, d_model, n_heads, dropout):
        super().__init__()

        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.dropout = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

    def split_heads(self, x):
        B, T, D = x.shape
        return x.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

    def forward(self, x, attn_mask):

        q, k, v = self.split_heads(self.q_proj(x)), self.split_heads(self.k_proj(x)), self.split_heads(self.v_proj(x))

        out = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask,
                                             dropout_p=self.dropout if self.training else 0.0)
        B, H, T, d = out.shape

        return self.out_proj(out.transpose(1, 2).reshape(B, T, H * d))


class PositionalEmbedding(nn.Module):

    def __init__(self, max_len, d_model, dropout):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return self.dropout(x + self.embedding(positions).unsqueeze(0))


class FeedForward(nn.Module):

    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.fc2(F.gelu(self.fc1(x))))


class EncoderBlock(nn.Module):

    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)
        self.norm2 = nn.LayerNorm(d_model)
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

    def forward(self, x, attn_mask):
        x = x + self.self_attention(self.norm1(x), attn_mask)
        return x + self.feed_forward(self.norm2(x))


class Arch2Encoder(nn.Module):
    """Architecture-2 encoder (the decoder is not needed): bidirectional, learned positions."""

    def __init__(self, cfg):
        super().__init__()

        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["d_model"])
        self.position_embedding = PositionalEmbedding(cfg["src_len"], cfg["d_model"], cfg["dropout"])
        self.layers = nn.ModuleList([
            EncoderBlock(cfg["d_model"], cfg["n_heads"], cfg["ffn_dim"], cfg["dropout"])
            for _ in range(cfg["encoder_layers"])
        ])
        self.final_norm = nn.LayerNorm(cfg["d_model"])

    def embed(self, input_ids):
        """Token + position embeddings (the first hidden state)."""
        return self.position_embedding(self.token_embedding(input_ids))

    def forward(self, input_ids, mask, all_layers=False):
        """all_layers=False: final hidden vectors (as in Phase 1).
        all_layers=True : list of the embeddings + the output of every layer (before any norm)."""

        attn_mask = mask[:, None, None, :]                    # padding is never attended to
        x = self.embed(input_ids)
        states = [x]

        for layer in self.layers:
            x = layer(x, attn_mask)
            states.append(x)

        if all_layers:
            return states

        return self.final_norm(x)


def backbone_blocks(backbone):
    """The transformer blocks of a Phase-1 model, bottom -> top
    (Architecture-2 calls them layers, Architecture-4 blocks)."""

    return list(backbone.layers if hasattr(backbone, "layers") else backbone.blocks)


def set_trainable_blocks(backbone, n_top):
    """Freeze every Phase-1 weight, then unfreeze ALL weights of the top n_top blocks (0 = none)."""

    for p in backbone.parameters():
        p.requires_grad = False

    blocks = backbone_blocks(backbone)
    for block in blocks[len(blocks) - n_top:]:
        for p in block.parameters():
            p.requires_grad = True


def block_groups(model, n_top):
    """(parameters, learning rate, weight decay) per trained block: top block LR_TOP_BLOCK,
    every block below LAYER_LR_DECAY x the rate of the block above it."""

    blocks = backbone_blocks(model.backbone)
    top = blocks[len(blocks) - n_top:] if n_top else []

    return [(list(block.parameters()), LR_TOP_BLOCK * LAYER_LR_DECAY ** k, 0.01) for k, block in enumerate(reversed(top))]


# ============================================================
# Standardization with RUNNING statistics (no learned weights)
#   x -> (x - mean) / std
# The statistics start from the frozen Phase-1 model (CELL 11) and then
# slowly follow the features while training changes them.
# ============================================================

class RunningStandardize(nn.Module):

    def __init__(self, dim, momentum=0.01):
        super().__init__()
        self.momentum = momentum
        self.register_buffer("mean", torch.zeros(dim))
        self.register_buffer("std", torch.ones(dim))

    def forward(self, x, mask):
        """x [..., dim], mask [...] (True = real value)."""

        if self.training:
            with torch.no_grad():
                values = x[mask]
                if len(values) > 1:
                    self.mean.mul_(1 - self.momentum).add_(self.momentum * values.mean(dim=0))
                    self.std.mul_(1 - self.momentum).add_(self.momentum * values.std(dim=0).clamp(min=1e-4))

        return (x - self.mean) / self.std


# ============================================================
# Chunks -> padded token ids, and which tokens are pooled
# ============================================================

SPACE_IDS = torch.tensor(SPACE_TOKEN_IDS if POOL_SKIP_SPACES and SPACE_TOKEN_IDS else [-1], dtype=torch.long, device=DEVICE)


def pad_chunks(chunks):
    """List of token-id arrays -> ids [n, T], mask [n, T] (True = real token), on the GPU."""

    max_len = max(len(c) for c in chunks)
    ids = torch.full((len(chunks), max_len), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(chunks), max_len), dtype=torch.bool)

    for row, c in enumerate(chunks):
        ids[row, :len(c)] = torch.from_numpy(c.astype(np.int64))
        mask[row, :len(c)] = True

    return ids.to(DEVICE), mask.to(DEVICE)


def pooling_mask(ids, mask):
    """Tokens that are pooled: real tokens, not whitespace, not the first POOL_SKIP_TOKENS of a chunk."""

    pool = mask & ~torch.isin(ids, SPACE_IDS)

    long_chunk = pool.sum(dim=1) > 2 * POOL_SKIP_TOKENS
    pool[long_chunk, :POOL_SKIP_TOKENS] = False

    empty = ~pool.any(dim=1)                   # a chunk with nothing left keeps all its real tokens
    pool[empty] = mask[empty]

    return pool


# ============================================================
# Document classifier (multi-label)
# ============================================================

class DocumentClassifier(nn.Module):

    def __init__(self, backbone, d_model, n_states):
        super().__init__()

        self.backbone = backbone
        self.d_model = d_model
        self.train_backbone = True          # False = the Phase-1 model runs without gradients (head-only epoch)

        # layer mix: token vector = sum_l w_l * LayerNorm(h_l), w = softmax(layer_logits)
        # (start: CELL 11, the best layers get the largest weights)
        self.layer_logits = nn.Parameter(torch.zeros(n_states))

        self.token_standardize = RunningStandardize(d_model)
        self.chunk_standardize = RunningStandardize(d_model)

        # LINEAR path: starts as the logistic regression of CELL 11
        self.linear_path = nn.Linear(d_model, NUM_LABELS)

        # LABEL ATTENTION: one query and one output vector per statute (output starts at 0)
        self.label_key = nn.Linear(d_model, LABEL_ATTENTION_DIM)
        self.label_query = nn.Parameter(0.02 * torch.randn(NUM_LABELS, LABEL_ATTENTION_DIM))
        self.label_output = nn.Parameter(torch.zeros(NUM_LABELS, d_model))
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        self.register_buffer("pos_weight", POS_WEIGHT.clone())
        self.kd_alpha = 0.0                 # > 0 only for the student (CELL 15)

    def backbone_parameters(self):
        return [p for p in self.backbone.parameters() if p.requires_grad]

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def layer_weights(self):
        return torch.softmax(self.layer_logits.float(), dim=0)

    def mixed_states(self, ids, mask):
        """[n, T] ids -> [n, T, D] float32: mix of the LayerNorm-ed hidden states of all layers."""

        with torch.set_grad_enabled(torch.is_grad_enabled() and self.train_backbone):
            with autocast():
                states = self.backbone(ids, mask, all_layers=True)

        mixed = 0.0
        for w, h in zip(self.layer_weights(), states):
            mixed = mixed + w * F.layer_norm(h.float(), h.shape[-1:])

        return mixed

    def forward(self, docs):
        """List of documents -> statute scores [n_docs, NUM_LABELS] (logits)."""

        chunks = [c for d in docs for c in d["chunks"]]
        ids, mask = pad_chunks(chunks)
        pool = pooling_mask(ids, mask)

        z = self.token_standardize(self.mixed_states(ids, mask), pool)             # [n_chunks, T, D]

        # chunk vector = mean of its pooled token vectors, standardized
        m = pool.unsqueeze(-1).float()
        chunk_vectors = self.chunk_standardize((z * m).sum(dim=1) / m.sum(dim=1),
                                               torch.ones(len(chunks), dtype=torch.bool, device=DEVICE))

        scores, k = [], 0

        for d in docs:
            n = len(d["chunks"])

            # LINEAR path: mean over the chunks of the document
            linear = self.linear_path(chunk_vectors[k:k + n].mean(dim=0))

            # LABEL ATTENTION over all pooled tokens of the document
            tokens = z[k:k + n][pool[k:k + n]]                                       # [N, D]
            keys = torch.tanh(self.label_key(tokens))                                # [N, A]
            attention = torch.softmax(keys @ self.label_query.T, dim=0)             # [N, statutes], sums to 1 over the tokens
            per_statute = self.dropout(attention.T @ tokens)                         # [statutes, D]

            scores.append(linear + (per_statute * self.label_output).sum(dim=-1))
            k += n

        return torch.stack(scores)

    def loss(self, docs):
        """Gold labels: BCE with the positive weights. Student: + knowledge distillation:
        (1 - KD_ALPHA) * BCE(gold) + KD_ALPHA * T^2 * BCE(student, teacher probabilities)."""

        logits = self(docs).float()
        y = torch.tensor(np.stack([d["y"] for d in docs]), device=DEVICE)
        hard = F.binary_cross_entropy_with_logits(logits, y, pos_weight=self.pos_weight)

        if self.kd_alpha == 0:
            return hard

        # match the teacher's probability of EVERY statute (soft targets, Hinton et al. 2015)
        T = KD_TEMPERATURE
        teacher_logits = torch.tensor(np.stack([d["teacher_logit"] for d in docs]), device=DEVICE)
        soft = F.binary_cross_entropy_with_logits(logits / T, torch.sigmoid(teacher_logits / T)) * T ** 2

        return (1 - self.kd_alpha) * hard + self.kd_alpha * soft


In [9]:
# ============================================================
# CELL 9 — LOAD BOTH PHASE-1 CHECKPOINTS + MODEL BUILDERS
# ============================================================

def find_checkpoint(path, file_name, required_key):
    """Use `path` if it exists, else search /kaggle/input for `file_name` whose config has `required_key`."""

    if Path(path).exists():
        return Path(path)

    for candidate in sorted(Path("/kaggle/input").rglob(file_name)):
        try:
            ckpt = torch.load(candidate, map_location="cpu", weights_only=False)
            if required_key in ckpt.get("model_config", ckpt.get("config", {})):
                print("Found:", candidate)
                return candidate
        except Exception:
            pass

    raise FileNotFoundError(f"{path} not found and no matching {file_name} under /kaggle/input")


# ---------------- teacher: Architecture-4 ----------------
TEACHER_CHECKPOINT = find_checkpoint(TEACHER_CHECKPOINT, "best.pt", "prefix_length")
phase1 = torch.load(TEACHER_CHECKPOINT, map_location="cpu", weights_only=False)

TEACHER_CONFIG = dict(phase1["model_config"])
TEACHER_CONFIG.setdefault("norm_eps", 1e-5)
TEACHER_CONFIG["dropout"] = BACKBONE_DROPOUT
if PREFIX_LENGTH is not None:
    TEACHER_CONFIG["prefix_length"] = PREFIX_LENGTH

# weights without the LM head (tied to the embedding, not needed here)
TEACHER_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

# ---------------- student: Architecture-2 encoder ----------------
STUDENT_CHECKPOINT = find_checkpoint(STUDENT_CHECKPOINT, "best_encoder_decoder.pt", "encoder_layers")
phase1 = torch.load(STUDENT_CHECKPOINT, map_location="cpu", weights_only=False)

STUDENT_CONFIG = dict(phase1["config"])
STUDENT_CONFIG["dropout"] = BACKBONE_DROPOUT

# encoder weights only (the decoder of Architecture-2 is not needed)
STUDENT_WEIGHTS = {}
for key, value in phase1["model_state_dict"].items():
    key = key.replace("_orig_mod.", "")
    if key.startswith("encoder."):
        STUDENT_WEIGHTS[key[len("encoder."):]] = value

del phase1

print("Teacher config:", {k: TEACHER_CONFIG[k] for k in ("d_model", "n_layers", "n_heads", "ffn_dim", "block_size", "prefix_length")})
print("Student config:", {k: STUDENT_CONFIG[k] for k in ("d_model", "encoder_layers", "n_heads", "ffn_dim", "src_len")})

for config in (TEACHER_CONFIG, STUDENT_CONFIG):
    assert config["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert CHUNK_LEN <= TEACHER_CONFIG["block_size"] and CHUNK_LEN <= STUDENT_CONFIG["src_len"]


def build(backbone, weights, d_model, n_top):
    """Phase-1 backbone (top n_top blocks trainable, everything else frozen) + new classifier."""

    backbone.load_state_dict(weights, strict=True)          # every name and shape must match
    set_trainable_blocks(backbone, n_top)

    # the hidden states of all layers; the last one (after the final norm) = the Phase-1 output
    test_ids = torch.randint(4, VOCAB_SIZE, (2, 24))
    test_mask = torch.ones(2, 24, dtype=torch.bool)
    test_mask[1, 16:] = False

    backbone.eval()
    with torch.no_grad():
        states = backbone(test_ids, test_mask, all_layers=True)
        assert len(states) == len(backbone_blocks(backbone)) + 1
        assert torch.allclose(backbone.final_norm(states[-1]), backbone(test_ids, test_mask), atol=1e-5)
    backbone.train()

    return DocumentClassifier(backbone, d_model, len(backbone_blocks(backbone)) + 1).to(DEVICE)


def build_teacher():
    return build(Arch4Hybrid(TEACHER_CONFIG), TEACHER_WEIGHTS, TEACHER_CONFIG["d_model"], 0)


def build_student():
    return build(Arch2Encoder(STUDENT_CONFIG), STUDENT_WEIGHTS, STUDENT_CONFIG["d_model"], STUDENT_TOP_BLOCKS)


# sizes (built on the CPU once, then freed)
_t, _s = Arch4Hybrid(TEACHER_CONFIG), Arch2Encoder(STUDENT_CONFIG)
TEACHER_BACKBONE_SIZE = sum(p.numel() for p in _t.parameters())
STUDENT_BACKBONE_SIZE = sum(p.numel() for p in _s.parameters())
assert 0 < STUDENT_TOP_BLOCKS <= len(backbone_blocks(_s)), f"STUDENT_TOP_BLOCKS must be 1..{len(backbone_blocks(_s))}"
del _t, _s

print(f"\nTeacher backbone: {TEACHER_BACKBONE_SIZE:,} weights | student backbone: {STUDENT_BACKBONE_SIZE:,} "
      f"({TEACHER_BACKBONE_SIZE / STUDENT_BACKBONE_SIZE:.1f}x smaller)")


# ------------------------------------------------------------
# Train split: the teacher learns from one part, scores another part for the student
# ------------------------------------------------------------

_order = list(range(len(train_docs)))
random.Random(SEED + 5).shuffle(_order)

teacher_train_docs = [train_docs[i] for i in _order[:TEACHER_TRAIN_DOCS]]
student_train_docs = [train_docs[i] for i in _order[TEACHER_TRAIN_DOCS:TEACHER_TRAIN_DOCS + STUDENT_TRAIN_DOCS]]

assert not {d["id"] for d in teacher_train_docs} & {d["id"] for d in student_train_docs} or \
    len({d["id"] for d in train_docs}) < len(train_docs)
print(f"train documents: teacher learns from {len(teacher_train_docs)}, student from {len(student_train_docs)} others")


Teacher config: {'d_model': 768, 'n_layers': 16, 'n_heads': 12, 'ffn_dim': 2048, 'block_size': 2048, 'prefix_length': 512}
Student config: {'d_model': 768, 'encoder_layers': 6, 'n_heads': 12, 'ffn_dim': 3072, 'src_len': 1024}

Teacher backbone: 205,496,064 weights | student backbone: 67,894,272 (3.0x smaller)
train documents: teacher learns from 16000, student from 16000 others


In [10]:
# ============================================================
# CELL 10 — PREDICTION + EVALUATION
#   documents are processed in groups of up to EVAL_CHUNKS chunks
# ============================================================

@torch.no_grad()
def predict_probs(model, docs):
    """[n_docs, NUM_LABELS] probabilities."""

    model.eval()

    probs, group, n_chunks = [], [], 0

    for d in docs + [None]:

        if group and (d is None or n_chunks + len(d["chunks"]) > EVAL_CHUNKS):
            probs.append(torch.sigmoid(model(group).float()).cpu())
            group, n_chunks = [], 0

        if d is not None:
            group.append(d)
            n_chunks += len(d["chunks"])

    model.train()

    return torch.cat(probs).numpy()


def evaluate(model, docs):
    """Macro-F1 etc. with thresholds tuned on these documents (used to pick the best epoch)."""

    Y = np.stack([d["y"] for d in docs])
    probs = predict_probs(model, docs)
    scores, thresholds, at_least_one = dev_scores(Y, probs)

    return scores, probs


In [11]:
# ============================================================
# CELL 11 — PREPARE A MODEL BEFORE TRAINING (teacher and student)
#
#   1. per-layer means (LayerNorm-ed token vectors, averaged over the chunks) of
#      PROBE_DOCS train documents and DEV_EVAL_DOCS dev documents (frozen Phase-1 model)
#   2. probe: a linear multi-label model on every layer -> dev macro-F1
#      -> the layer mix STARTS on the best layers
#   3. token and chunk statistics of the mixed vectors -> stored in the model
#   4. the LINEAR path starts as a logistic regression (same loss as training:
#      BCE with the positive weights) on exactly the features it sees
# ============================================================

dev_eval_docs = random.Random(SEED + 3).sample(dev_docs, min(DEV_EVAL_DOCS, len(dev_docs)))
Y_dev_eval = np.stack([d["y"] for d in dev_eval_docs])


def chunk_batches(chunks):
    """Batches of EVAL_CHUNKS chunks: (ids, mask, pooling mask) on the GPU."""

    for k in range(0, len(chunks), EVAL_CHUNKS):
        ids, mask = pad_chunks(chunks[k:k + EVAL_CHUNKS])
        yield ids, mask, pooling_mask(ids, mask)


@torch.no_grad()
def layer_means(model, docs):
    """[n_docs, N_STATES, D]: per document and layer, the mean LayerNorm-ed token vector (averaged over its chunks)."""

    model.eval()

    chunks = [c for d in docs for c in d["chunks"]]
    per_chunk = []

    for ids, mask, pool in chunk_batches(chunks):
        with autocast():
            states = model.backbone(ids, mask, all_layers=True)
        m = pool.unsqueeze(-1)
        per_chunk.append(torch.stack(
            [(F.layer_norm(h.float(), h.shape[-1:]) * m).sum(dim=1) / m.sum(dim=1) for h in states], dim=1).cpu())

    model.train()

    per_chunk = torch.cat(per_chunk)
    out, k = [], 0
    for d in docs:
        n = len(d["chunks"])
        out.append(per_chunk[k:k + n].mean(dim=0))
        k += n

    return torch.stack(out)


def fit_linear(X, Y, weight_decay, steps=300):
    """Multi-label logistic regression on fixed features X [n, D] (full batch, Adam), with the
    training loss (BCE + positive weights) + L2. Returns W [D, L], b [L]."""

    X = torch.as_tensor(X, dtype=torch.float32, device=DEVICE)
    Y = torch.as_tensor(Y, dtype=torch.float32, device=DEVICE)
    W = torch.zeros(X.shape[1], Y.shape[1], device=DEVICE, requires_grad=True)
    b = torch.logit(Y.mean(dim=0).clamp(1e-3, 1 - 1e-3)).clone().requires_grad_(True)

    optimizer = torch.optim.Adam([W, b], lr=0.01)
    pos_weight = POS_WEIGHT.to(DEVICE)

    with torch.enable_grad():
        for _ in range(steps):
            loss = F.binary_cross_entropy_with_logits(X @ W + b, Y, pos_weight=pos_weight) + weight_decay * W.pow(2).sum()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    return W.detach(), b.detach()


def linear_probs(X, W, b):
    return torch.sigmoid(torch.as_tensor(X, dtype=torch.float32, device=DEVICE) @ W + b).cpu().numpy()


def prepare(model, name, train_pool):
    """Steps 1-4 for one model (teacher or student). train_pool = the train documents it learns from."""

    t0 = time.time()

    N_STATES = len(model.layer_logits)
    LAYER_NAMES = ["embeddings"] + [f"layer {i}" for i in range(1, N_STATES)]
    D_MODEL = model.d_model

    print(f"\n==================== {name} ====================")

    # ---------------- 1. per-layer means ----------------
    probe_docs = random.Random(SEED + 1).sample(train_pool, min(PROBE_DOCS, len(train_pool)))

    L_probe = layer_means(model, probe_docs)
    L_dev = layer_means(model, dev_eval_docs)
    Y_probe = np.stack([d["y"] for d in probe_docs])
    Y_dev_eval = np.stack([d["y"] for d in dev_eval_docs])

    print(f"per-layer means of {len(probe_docs)} train + {len(dev_eval_docs)} dev documents | {time.time() - t0:.0f}s")


    # ---------------- 2. probe every layer ----------------
    print(f"\n  {'layer':12s} {'dev macro-F1':>13s} {'micro-F1':>9s}")
    probe_results = {}

    for l, name in enumerate(LAYER_NAMES):

        mu = L_probe[:, l].mean(dim=0)
        sd = L_probe[:, l].std(dim=0).clamp(min=1e-6)
        W, b = fit_linear((L_probe[:, l] - mu) / sd, Y_probe, weight_decay=1e-4, steps=200)
        scores, _, _ = dev_scores(Y_dev_eval, linear_probs((L_dev[:, l] - mu) / sd, W, b))

        probe_results[name] = {"macro_f1": scores["macro_f1"], "micro_f1": scores["micro_f1"]}
        print(f"  {name:12s} {scores['macro_f1']:13.4f} {scores['micro_f1']:9.4f}")

    PROBE_BEST_LAYER = max(probe_results, key=lambda n: probe_results[n]["macro_f1"])
    PROBE_BEST = probe_results[PROBE_BEST_LAYER]

    # the layer mix starts on the best layers: weight ~ exp((F1 - best F1) / 0.02), so layers that
    # are almost as good share the weight and much worse layers start near 0 (the mix is learned)
    with torch.no_grad():
        model.layer_logits.copy_(torch.tensor(
            [(probe_results[n]["macro_f1"] - PROBE_BEST["macro_f1"]) / 0.02 for n in LAYER_NAMES]).to(DEVICE))

    start_mix = model.layer_weights().detach().cpu()
    print(f"\nbest frozen layer: {PROBE_BEST_LAYER} (dev macro-F1 {PROBE_BEST['macro_f1']:.4f})")
    print("layer mix at the start:", {n: round(float(w), 3) for n, w in zip(LAYER_NAMES, start_mix)})


    # ---------------- 3. token + chunk statistics of the mixed vectors ----------------
    stat_docs = random.Random(SEED).sample(train_pool, min(STATS_DOCS, len(train_pool)))
    stat_chunks = [c for d in stat_docs for c in d["chunks"]]

    total = torch.zeros(D_MODEL, dtype=torch.float64, device=DEVICE)
    total_sq = torch.zeros_like(total)
    count = 0

    model.eval()
    with torch.no_grad():

        for ids, mask, pool in chunk_batches(stat_chunks):
            hidden = model.mixed_states(ids, mask)
            m = pool.unsqueeze(-1)
            total += (hidden * m).sum(dim=(0, 1)).double()
            total_sq += (hidden.pow(2) * m).sum(dim=(0, 1)).double()
            count += int(pool.sum())

        TOKEN_MEAN = (total / count).float()
        TOKEN_STD = (total_sq / count - (total / count).pow(2)).clamp(min=1e-8).sqrt().float().clamp(min=1e-4)
        model.token_standardize.mean.copy_(TOKEN_MEAN)
        model.token_standardize.std.copy_(TOKEN_STD)

        chunk_means = []
        for ids, mask, pool in chunk_batches(stat_chunks):
            z = model.token_standardize(model.mixed_states(ids, mask), pool)
            m = pool.unsqueeze(-1).float()
            chunk_means.append(((z * m).sum(dim=1) / m.sum(dim=1)).cpu())

        chunk_means = torch.cat(chunk_means)
        CHUNK_MEAN = chunk_means.mean(dim=0)
        CHUNK_STD = chunk_means.std(dim=0).clamp(min=1e-4)
        model.chunk_standardize.mean.copy_(CHUNK_MEAN.to(DEVICE))
        model.chunk_standardize.std.copy_(CHUNK_STD.to(DEVICE))

    model.train()

    print(f"\nfeature statistics from {count:,} pooled tokens of {len(stat_docs)} train documents")


    # ---------------- 4. the linear path starts as a logistic regression ----------------
    # document feature = mean over its chunks of the standardized chunk means of the MIXED token
    # vectors (exactly what the linear path sees; everything is linear, so it follows from the
    # per-layer means of step 1)

    def linear_path_features(L):
        mixed = (L * start_mix[None, :, None]).sum(dim=1)
        z = (mixed - TOKEN_MEAN.cpu()) / TOKEN_STD.cpu()
        return (z - CHUNK_MEAN) / CHUNK_STD


    F_probe, F_dev = linear_path_features(L_probe), linear_path_features(L_dev)

    best = None
    for weight_decay in (1e-6, 1e-5, 1e-4, 1e-3):
        W, b = fit_linear(F_probe, Y_probe, weight_decay)
        scores, _, _ = dev_scores(Y_dev_eval, linear_probs(F_dev, W, b))
        print(f"  logistic regression (L2 {weight_decay:g}): dev macro-F1 {scores['macro_f1']:.4f}, micro-F1 {scores['micro_f1']:.4f}")
        if best is None or scores["macro_f1"] > best[0]["macro_f1"]:
            best = (scores, W, b)

    START, W, b = best

    with torch.no_grad():
        model.linear_path.weight.copy_(W.T)
        model.linear_path.bias.copy_(b)

    print(f"\n{name}: the classifier STARTS as this logistic regression: dev macro-F1 {START['macro_f1']:.4f} "
          f"| prepared in {time.time() - t0:.0f}s")

    return {"layer_names": LAYER_NAMES, "probe": probe_results, "best_layer": PROBE_BEST_LAYER,
            "start": {k: v for k, v in START.items() if k != "per_statute_f1"}}


In [12]:
# ============================================================
# CELL 12 — OPTIMIZER + TRAINING FUNCTION (teacher and student)
#   groups: the trained Phase-1 blocks (none for the teacher), then the head (last group)
#   epoch 0 = the start (kept if no epoch beats it), head-only epochs, then blocks + head
# ============================================================

def make_optimizer(model, backbone_groups, total_steps, head_only_steps):

    groups = [{"params": params, "lr": lr, "weight_decay": wd} for params, lr, wd in backbone_groups]
    groups.append({"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.01})

    optimizer = torch.optim.AdamW(groups, betas=(0.9, 0.999))

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0. The Phase-1 part waits for head_only_steps, then has its own warmup."""

        decay = 1.0 if step < warmup_steps else max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        head_factor = min(1.0, (step + 1) / warmup_steps) * decay
        backbone_factor = min(1.0, max(0, step + 1 - head_only_steps) / warmup_steps) * decay

        for group, base_lr in zip(optimizer.param_groups[:-1], base_lrs[:-1]):
            group["lr"] = base_lr * backbone_factor
        optimizer.param_groups[-1]["lr"] = base_lrs[-1] * head_factor

    return optimizer, set_lr


def trainable_state(model):
    """The trained weights + the running feature statistics (to go back to the best epoch)."""

    state = {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}
    state.update({name: b.detach().cpu().clone() for name, b in model.named_buffers() if "standardize" in name})

    return state


def train_model(model, name, docs, epochs, epoch_docs, batch_docs, grad_accum, head_only_epochs, backbone_groups):
    """
    Trains on the given TRAIN documents only. Every epoch uses epoch_docs documents (shuffled once
    and walked through). After every epoch: macro-F1 on DEV_EVAL_DOCS dev documents (thresholds
    tuned on them); the best epoch is kept.
    """

    docs_per_epoch = min(epoch_docs, len(docs))
    updates_per_epoch = math.ceil(docs_per_epoch / (batch_docs * grad_accum))

    optimizer, set_lr = make_optimizer(model, backbone_groups, epochs * updates_per_epoch, head_only_epochs * updates_per_epoch)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    backbone_params = [p for group in optimizer.param_groups[:-1] for p in group["params"]]
    head_params = optimizer.param_groups[-1]["params"]

    rng = random.Random(SEED)
    order = list(range(len(docs)))
    rng.shuffle(order)
    cursor = 0

    history = []
    step = 0
    t_start = time.time()

    # epoch 0 = the start (logistic regression on the frozen features): kept if no epoch beats it
    scores, _ = evaluate(model, dev_eval_docs)
    best_score, best_epoch, best_state = scores["macro_f1"], 0, trainable_state(model)
    history.append({"epoch": 0, "dev": {k: v for k, v in scores.items() if k != "per_statute_f1"}})
    print(f"{name} epoch 0 (start) | dev macro-F1 {scores['macro_f1']:.4f} | micro-F1 {scores['micro_f1']:.4f} | "
          f"subset acc {scores['subset_acc']:.4f}", flush=True)

    model.train()

    for epoch in range(1, epochs + 1):

        if cursor + docs_per_epoch > len(order):
            rng.shuffle(order)
            cursor = 0

        epoch_batch_docs = [docs[i] for i in order[cursor:cursor + docs_per_epoch]]
        cursor += docs_per_epoch

        # head-only epochs: the Phase-1 model runs without gradients (its blocks do not train yet)
        model.train_backbone = epoch > head_only_epochs

        batches = [epoch_batch_docs[k:k + batch_docs] for k in range(0, len(epoch_batch_docs), batch_docs)]

        total_loss, skipped = 0.0, 0
        optimizer.zero_grad(set_to_none=True)

        for b, batch in enumerate(batches):

            loss = model.loss(batch)
            scaler.scale(loss / grad_accum).backward()
            total_loss += loss.item()

            if (b + 1) % grad_accum == 0 or b + 1 == len(batches):

                set_lr(step)
                scaler.unscale_(optimizer)

                # clip the Phase-1 part and the head separately
                torch.nn.utils.clip_grad_norm_(backbone_params, GRAD_CLIP)
                torch.nn.utils.clip_grad_norm_(head_params, GRAD_CLIP)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                skipped += scaler.get_scale() < scale_before

                optimizer.zero_grad(set_to_none=True)
                step += 1

            if (b + 1) % 1000 == 0:
                print(f"    batch {b + 1}/{len(batches)} | loss {total_loss / (b + 1):.4f} | "
                      f"{(time.time() - t_start) / 60:.1f} min", flush=True)

        scores, _ = evaluate(model, dev_eval_docs)
        history.append({"epoch": epoch, "loss": total_loss / len(batches),
                        "dev": {k: v for k, v in scores.items() if k != "per_statute_f1"}, "skipped_steps": int(skipped)})

        line = (f"{name} epoch {epoch}/{epochs}{' (head only)' if epoch <= head_only_epochs else ''} | loss {total_loss / len(batches):.4f} | "
                f"dev macro-F1 {scores['macro_f1']:.4f} | micro-F1 {scores['micro_f1']:.4f} | subset acc {scores['subset_acc']:.4f}")

        stop = False
        if scores["macro_f1"] > best_score:
            best_score, best_epoch, best_state = scores["macro_f1"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        print(line + f" | {(time.time() - t_start) / 60:.1f} min", flush=True)

        mix = " ".join(f"{w:.2f}" for w in model.layer_weights().detach().tolist())
        print(f"    layer mix (embeddings, layer 1..): {mix}")

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state, strict=False)          # back to the best epoch
    model.train_backbone = True

    return history, best_epoch


In [13]:
# ============================================================
# CELL 13 — STEP 1: THE TEACHER (Architecture-4), trained on its TRAIN documents with the gold statutes
#   the 205M Phase-1 model stays frozen; the classifier trains
# ============================================================

for name in ["teacher", "model", "optimizer", "scaler"]:
    globals().pop(name, None)
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

teacher = build_teacher()
assert not any(p.requires_grad for p in teacher.backbone.parameters())
TEACHER_PREP = prepare(teacher, "Teacher", teacher_train_docs)

print(f"\nTeacher | train {len(teacher_train_docs)} documents ({min(TEACHER_EPOCH_DOCS, len(teacher_train_docs))} per epoch) "
      f"-> dev {len(dev_eval_docs)} documents after every epoch")

t0 = time.time()
teacher_history, teacher_best_epoch = train_model(
    teacher, "Teacher", teacher_train_docs, TEACHER_EPOCHS, TEACHER_EPOCH_DOCS, TEACHER_BATCH_DOCS,
    TEACHER_GRAD_ACCUM, head_only_epochs=TEACHER_EPOCHS, backbone_groups=[])
teacher_minutes = (time.time() - t0) / 60
teacher_peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

TEACHER_DEV_EVAL_PROBS = predict_probs(teacher, dev_eval_docs)
teacher_scores, _, _ = dev_scores(Y_dev_eval, TEACHER_DEV_EVAL_PROBS)

print(f"\n✅ Teacher: dev macro-F1 {teacher_scores['macro_f1']:.4f} | micro-F1 {teacher_scores['micro_f1']:.4f} "
      f"({len(dev_eval_docs)} dev documents) | best epoch {teacher_best_epoch} | {teacher_minutes:.1f} min | peak GPU {teacher_peak_gb:.2f} GB")



==================== Teacher ====================
per-layer means of 3000 train + 2000 dev documents | 406s

  layer         dev macro-F1  micro-F1
  embeddings          0.2424    0.3178
  layer 1             0.1555    0.2055
  layer 2             0.1563    0.2058
  layer 3             0.1546    0.2136
  layer 4             0.1511    0.2072
  layer 5             0.1507    0.2150
  layer 6             0.1499    0.2073
  layer 7             0.1500    0.2073
  layer 8             0.1499    0.2054
  layer 9             0.1468    0.1948
  layer 10            0.1469    0.1961
  layer 11            0.1459    0.1994
  layer 12            0.1457    0.2026
  layer 13            0.1446    0.1932
  layer 14            0.1430    0.1946
  layer 15            0.1389    0.1905
  layer 16            0.1314    0.1763

best frozen layer: embeddings (dev macro-F1 0.2424)

feature statistics from 510,972 pooled tokens of 500 train documents
  logistic regression (L2 1e-06): dev macro-F1 0.2095, micro-F1 0

In [14]:
# ============================================================
# CELL 14 — STEP 2: TEACHER PROBABILITIES
#   the student's train documents (the teacher did NOT learn from them)
#   -> doc["teacher_logit"] = the teacher's score of every statute
#   then the teacher is removed from the GPU
# ============================================================

t0 = time.time()

for k in range(0, len(student_train_docs), 2000):

    part = student_train_docs[k:k + 2000]
    probs = np.clip(predict_probs(teacher, part), 1e-6, 1 - 1e-6)

    for d, p in zip(part, probs):
        d["teacher_logit"] = np.log(p / (1 - p)).astype(np.float32)

    print(f"  teacher scored {min(k + 2000, len(student_train_docs))}/{len(student_train_docs)} train documents | "
          f"{(time.time() - t0) / 60:.1f} min", flush=True)

Y_student = np.stack([d["y"] for d in student_train_docs])
P_student = 1 / (1 + np.exp(-np.stack([d["teacher_logit"] for d in student_train_docs])))
teacher_on_student_docs = multilabel_scores(Y_student, decide(P_student, tune_thresholds(Y_student, P_student), True))

print(f"\nTeacher on the student's train documents (unseen by the teacher): macro-F1 {teacher_on_student_docs['macro_f1']:.4f} "
      f"| mean probability of a gold statute {P_student[Y_student == 1].mean():.3f}, of the others {P_student[Y_student == 0].mean():.3f}")

torch.save(trainable_state(teacher), OUTPUT_DIR / "teacher_trainable.pt")

# the teacher is not needed any more: free the GPU for the student
del teacher
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


  teacher scored 2000/16000 train documents | 2.7 min
  teacher scored 4000/16000 train documents | 5.4 min
  teacher scored 6000/16000 train documents | 8.2 min
  teacher scored 8000/16000 train documents | 10.9 min
  teacher scored 10000/16000 train documents | 13.6 min
  teacher scored 12000/16000 train documents | 16.3 min
  teacher scored 14000/16000 train documents | 19.1 min
  teacher scored 16000/16000 train documents | 21.8 min



In [15]:
# ============================================================
# CELL 15 — STEP 3: THE STUDENT (Architecture-2 encoder) WITH KNOWLEDGE DISTILLATION
#   loss = (1 - KD_ALPHA) * BCE(gold) + KD_ALPHA * T^2 * BCE(student, teacher probabilities)
# ============================================================

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

model = build_student()                 # `model` = the student (used by the report and the test predictions)
model.kd_alpha = KD_ALPHA
STUDENT_PREP = prepare(model, "Student", student_train_docs)

n_total = sum(p.numel() for p in model.parameters())
n_trained = sum(p.numel() for p in model.backbone_parameters())
n_head = sum(p.numel() for p in model.head_parameters())
print(f"\nstudent: Phase-1 encoder {STUDENT_BACKBONE_SIZE:,} weights, {n_trained:,} of them train "
      f"(top {STUDENT_TOP_BLOCKS} layers) | head {n_head:,}")

print(f"\nStudent | train {len(student_train_docs)} documents with teacher probabilities "
      f"({min(STUDENT_EPOCH_DOCS, len(student_train_docs))} per epoch) -> dev {len(dev_eval_docs)}")

t0 = time.time()
history, best_epoch = train_model(
    model, "Student", student_train_docs, STUDENT_EPOCHS, STUDENT_EPOCH_DOCS, STUDENT_BATCH_DOCS,
    STUDENT_GRAD_ACCUM, STUDENT_HEAD_ONLY_EPOCHS, block_groups(model, STUDENT_TOP_BLOCKS))
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

# names used by the dev report (CELL 16)
START = STUDENT_PREP["start"]
probe_results = STUDENT_PREP["probe"]
PROBE_BEST_LAYER = STUDENT_PREP["best_layer"]
LAYER_NAMES = STUDENT_PREP["layer_names"]

print(f"\n✅ Student trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")



==================== Student ====================
per-layer means of 3000 train + 2000 dev documents | 97s

  layer         dev macro-F1  micro-F1
  embeddings          0.2320    0.3227
  layer 1             0.2091    0.2870
  layer 2             0.2127    0.2738
  layer 3             0.2172    0.2716
  layer 4             0.2221    0.2842
  layer 5             0.2249    0.2934
  layer 6             0.2250    0.2887

best frozen layer: embeddings (dev macro-F1 0.2320)
layer mix at the start: {'embeddings': 0.238, 'layer 1': 0.076, 'layer 2': 0.091, 'layer 3': 0.114, 'layer 4': 0.146, 'layer 5': 0.167, 'layer 6': 0.168}

feature statistics from 519,095 pooled tokens of 500 train documents
  logistic regression (L2 1e-06): dev macro-F1 0.2303, micro-F1 0.3131
  logistic regression (L2 1e-05): dev macro-F1 0.2372, micro-F1 0.3230
  logistic regression (L2 0.0001): dev macro-F1 0.2375, micro-F1 0.3226
  logistic regression (L2 0.001): dev macro-F1 0.2253, micro-F1 0.2919

layer 6: the cla

In [16]:
# ============================================================
# CELL 16 — DEV REPORT (the STUDENT, all dev documents) + TEACHER vs STUDENT
#   thresholds per statute tuned on the full dev set (used for the test predictions),
#   scores, F1 by statute frequency, most missed / most over-predicted statutes
# ============================================================

t0 = time.time()
dev_probs = predict_probs(model, dev_docs)
predict_seconds = time.time() - t0

Y_dev = np.stack([d["y"] for d in dev_docs])

DEV_SCORES, THRESHOLDS, AT_LEAST_ONE = dev_scores(Y_dev, dev_probs)
dev_pred = decide(dev_probs, THRESHOLDS, AT_LEAST_ONE)

at_05 = multilabel_scores(Y_dev, decide(dev_probs, np.full(NUM_LABELS, 0.5), False))

print(f"DEV ({RUN_NAME}, best epoch {best_epoch}, {len(dev_docs)} documents, predicted in {predict_seconds:.0f}s)")
print(f"{'decision':38s} {'macro-F1':>9s} {'micro-F1':>9s} {'subset acc':>11s} {'macro-P':>8s} {'macro-R':>8s}")
print(f"{'threshold 0.5 for every statute':38s} {at_05['macro_f1']:9.4f} {at_05['micro_f1']:9.4f} {at_05['subset_acc']:11.4f} "
      f"{at_05['macro_precision']:8.4f} {at_05['macro_recall']:8.4f}")
print(f"{'threshold per statute' + (' + at least one' if AT_LEAST_ONE else ''):38s} {DEV_SCORES['macro_f1']:9.4f} "
      f"{DEV_SCORES['micro_f1']:9.4f} {DEV_SCORES['subset_acc']:11.4f} {DEV_SCORES['macro_precision']:8.4f} {DEV_SCORES['macro_recall']:8.4f}")
print(f"reference: start (logistic regression) {START['macro_f1']:.4f} on {len(dev_eval_docs)} dev docs | "
      f"TF-IDF word counts 0.394")

print(f"\nstatutes per document: gold {Y_dev.sum(axis=1).mean():.2f} | predicted {dev_pred.sum(axis=1).mean():.2f}")


# ------------------------------------------------------------
# F1 by how often the statute occurs in train
# ------------------------------------------------------------

per_f1 = DEV_SCORES["per_statute_f1"]
print(f"\n{'train docs per statute':>23s} {'statutes':>9s} {'mean F1':>8s}")
by_frequency = {}
for lo, hi in [(0, 499), (500, 999), (1000, 2999), (3000, 10 ** 9)]:
    group = [j for j in range(NUM_LABELS) if lo <= POSITIVES[j] <= hi]
    if group:
        name = f"{lo}-{hi}" if hi < 10 ** 9 else f">= {lo}"
        by_frequency[name] = {"statutes": len(group), "mean_f1": float(per_f1[group].mean())}
        print(f"{name:>23s} {len(group):9d} {per_f1[group].mean():8.4f}")


# ------------------------------------------------------------
# Every statute: F1, threshold, gold and predicted counts on dev
# ------------------------------------------------------------

tp = (Y_dev.astype(bool) & dev_pred).sum(axis=0)
gold = Y_dev.sum(axis=0).astype(int)
pred = dev_pred.sum(axis=0)

print(f"\n{'statute':>8s} {'train':>7s} {'dev gold':>9s} {'predicted':>10s} {'F1':>6s} {'threshold':>10s}")
for j in np.argsort(-POSITIVES):
    print(f"{LABELS[j]:8d} {int(POSITIVES[j]):7d} {gold[j]:9d} {int(pred[j]):10d} {per_f1[j]:6.3f} {THRESHOLDS[j]:10.2f}")

worst = np.argsort(per_f1)[:10]
print("\n10 worst statutes:", [(LABELS[j], round(float(per_f1[j]), 3)) for j in worst])

fn = (Y_dev.astype(bool) & ~dev_pred).sum(axis=0)
fp = (~Y_dev.astype(bool) & dev_pred).sum(axis=0)
print("most missed (false negatives):", [(LABELS[j], int(fn[j])) for j in np.argsort(-fn)[:10]])
print("most over-predicted (false positives):", [(LABELS[j], int(fp[j])) for j in np.argsort(-fp)[:10]])


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "KD student: Architecture-2 encoder (teacher Architecture-4); mix of all layers; linear path (starts as logistic regression) + label attention over all tokens",
    "trained_on": "train only (dev used to pick the best epochs and the thresholds); teacher and student on different train documents",
    "input": {"chunk_len": CHUNK_LEN, "max_chunks": MAX_CHUNKS, "placeholders": PLACEHOLDERS, "pool_skip_spaces": POOL_SKIP_SPACES},
    "technique": {"method": "Knowledge distillation", "teacher": "Architecture-4 (frozen) + classifier", "student": "Architecture-2 encoder",
                  "alpha": KD_ALPHA, "temperature": KD_TEMPERATURE, "student_top_blocks": STUDENT_TOP_BLOCKS,
                  "lr_top_block": LR_TOP_BLOCK, "layer_lr_decay": LAYER_LR_DECAY},
    "loss": {"bce_pos_weight_power": POS_WEIGHT_POWER, "pos_weight_max": POS_WEIGHT_MAX},
    "parameters_total": n_total, "parameters_trained_in_phase1_model": n_trained, "parameters_head": n_head,
    "best_epoch": best_epoch, "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "probe_by_layer": probe_results, "probe_best_layer": PROBE_BEST_LAYER,
    "start_logistic_regression": {k: v for k, v in START.items() if k != "per_statute_f1"},
    "layer_mix": {n: round(float(w), 4) for n, w in zip(LAYER_NAMES, model.layer_weights().detach())},
    "dev_threshold_0.5": {k: v for k, v in at_05.items() if k != "per_statute_f1"},
    "dev": {k: v for k, v in DEV_SCORES.items() if k != "per_statute_f1"},
    "at_least_one": AT_LEAST_ONE,
    "dev_by_frequency": by_frequency,
    "per_statute": {int(LABELS[j]): {"f1": float(per_f1[j]), "threshold": float(THRESHOLDS[j]),
                                     "train_docs": int(POSITIVES[j]), "dev_gold": int(gold[j]), "dev_predicted": int(pred[j])}
                    for j in range(NUM_LABELS)},
    "history": history,
}

# ------------------------------------------------------------
# Teacher vs student (same dev documents, thresholds tuned on them; only the STUDENT is submitted)
# ------------------------------------------------------------

dev_eval_index = {d["id"]: i for i, d in enumerate(dev_docs)}
student_eval_probs = dev_probs[[dev_eval_index[d["id"]] for d in dev_eval_docs]]
student_scores, _, _ = dev_scores(Y_dev_eval, student_eval_probs)

kd_comparison = {
    "teacher": {"model": "Architecture-4 hybrid prefix-LM (frozen) + classifier", "backbone_weights": TEACHER_BACKBONE_SIZE,
                "dev_eval_macro_f1": teacher_scores["macro_f1"], "dev_eval_micro_f1": teacher_scores["micro_f1"],
                "train_minutes": teacher_minutes, "peak_gpu_gb": teacher_peak_gb, "best_epoch": teacher_best_epoch,
                "probe_best_layer": TEACHER_PREP["best_layer"], "start": TEACHER_PREP["start"],
                "on_student_train_docs_macro_f1": teacher_on_student_docs["macro_f1"], "history": teacher_history},
    "student": {"model": "Architecture-2 encoder + classifier", "backbone_weights": STUDENT_BACKBONE_SIZE,
                "dev_eval_macro_f1": student_scores["macro_f1"], "dev_eval_micro_f1": student_scores["micro_f1"],
                "train_minutes": train_minutes, "peak_gpu_gb": peak_gpu_gb, "best_epoch": best_epoch},
    "dev_eval_docs": len(dev_eval_docs),
}

print(f"\nTEACHER vs STUDENT on the same {len(dev_eval_docs)} dev documents")
print(f"{'':10s} {'weights':>13s} {'macro-F1':>9s} {'micro-F1':>9s} {'train min':>10s}")
for role in ("teacher", "student"):
    r = kd_comparison[role]
    print(f"{role:10s} {r['backbone_weights']:13,d} {r['dev_eval_macro_f1']:9.4f} {r['dev_eval_micro_f1']:9.4f} {r['train_minutes']:10.1f}")
print(f"student = {STUDENT_BACKBONE_SIZE / TEACHER_BACKBONE_SIZE:.0%} of the teacher's weights | "
      f"keeps {student_scores['macro_f1'] / max(teacher_scores['macro_f1'], 1e-9):.0%} of its macro-F1")

results["knowledge_distillation"] = kd_comparison

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

np.save(OUTPUT_DIR / "dev_probabilities.npy", dev_probs.astype(np.float32))

print("\nSaved:", OUTPUT_DIR / "results.json", "and dev_probabilities.npy")


DEV (p5_kd_arch4_to_arch2, best epoch 4, 10181 documents, predicted in 212s)
decision                                macro-F1  micro-F1  subset acc  macro-P  macro-R
threshold 0.5 for every statute           0.2282    0.3373      0.0181   0.2235   0.2533
threshold per statute                     0.2606    0.3426      0.0046   0.2261   0.3553

statutes per document: gold 3.92 | predicted 7.24

 train docs per statute  statutes  mean F1
                  0-499        30   0.1793
                500-999        27   0.2002
              1000-2999        27   0.3028
                >= 3000        16   0.4440

 statute   train  dev gold  predicted     F1  threshold
       5   11730      2936       4710  0.564       0.34
      34   10380      2596       2504  0.753       0.60
      97    7880      1970       4293  0.446       0.28
      43    7728      1932       3631  0.415       0.30
      39    7120      1779       3485  0.427       0.32
      12    4324      1086       1848  0.353       0

In [17]:
# ============================================================
# CELL 17 — PREDICT TEST WITH THE STUDENT + WRITE prediction.json + submission.zip
#   format: [{"id": "case_001", "pred": [1, 7, 23]}, ...]  (integers, unique, valid statute ids)
# ============================================================

t0 = time.time()
test_probs = predict_probs(model, test_docs)
test_pred = decide(test_probs, THRESHOLDS, AT_LEAST_ONE)
print(f"Predicted {len(test_docs)} test documents in {time.time() - t0:.0f}s")

submission = [{"id": d["id"], "pred": [int(LABELS[j]) for j in np.where(row)[0]]}
              for d, row in zip(test_docs, test_pred)]


# ------------------------------------------------------------
# Format checks
# ------------------------------------------------------------

valid = set(int(l) for l in LABELS)

assert len(submission) == len(test_docs)
assert [s["id"] for s in submission] == [d["id"] for d in test_docs]
assert all(isinstance(p, int) and p in valid for s in submission for p in s["pred"])
assert all(len(s["pred"]) == len(set(s["pred"])) for s in submission)

n_pred = np.array([len(s["pred"]) for s in submission])
print(f"✅ {len(submission)} predictions | statutes per document: mean {n_pred.mean():.2f}, "
      f"none {int((n_pred == 0).sum())}, max {n_pred.max()}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(submission, f)

np.save(OUTPUT_DIR / "test_probabilities.npy", test_probs.astype(np.float32))

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("submission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)


Predicted 13019 test documents in 267s
✅ 13019 predictions | statutes per document: mean 7.07, none 130, max 36
submission.zip: ['prediction.json']
Saved: /kaggle/working/p5_kd_arch4_to_arch2/submission.zip
